In [1]:
import pandas as pd

spot = pd.read_parquet("data/module1_spot_v0.parquet")
spot = spot.sort_values("ts").set_index("ts")

futures = (
    pd.read_parquet("data/module1_futures_ohlcv_v0.parquet")
      .sort_values("ts")
      .set_index("ts")
)

funding = (
    pd.read_parquet("data/module1_funding_v0.parquet")
      .sort_values("ts")
      .set_index("ts")
)

oi = (
    pd.read_parquet("data/module1_open_interest_v0.parquet")
      .sort_values("ts")
      .set_index("ts")
)


In [2]:
close = spot["close"]
returns = (close / close.shift(1)).apply("log").dropna()
vol = returns.ewm(span=20).std().dropna()


In [3]:
common_index = close.index.intersection(returns.index).intersection(vol.index)

close = close.loc[common_index]
returns = returns.loc[common_index]
vol = vol.loc[common_index]


In [4]:
import numpy as np
import pandas as pd

def get_events(close, vol, pt_sl=(1,1), max_holding=20):
    """
    close: pd.Series of prices
    vol: pd.Series of volatility
    returns: events DataFrame with start/end times
    """
    events = []

    for t in range(len(close) - max_holding):
        start = close.index[t]
        p0 = close.iloc[t]
        sigma = vol.iloc[t]

        pt = pt_sl[0] * sigma
        sl = -pt_sl[1] * sigma

        for k in range(1, max_holding + 1):
            r = (close.iloc[t + k] / p0) - 1
            if r >= pt or r <= sl:
                events.append((start, close.index[t + k]))
                break
        else:
            events.append((start, close.index[t + max_holding]))

    return pd.DataFrame(events, columns=["t0", "t1"])


In [5]:
def get_labels(events, close):
    labels = []
    for _, row in events.iterrows():
        r = (close.loc[row.t1] / close.loc[row.t0]) - 1
        labels.append(np.sign(r))
    return pd.Series(labels, index=events.index)


In [6]:
def get_concurrency(events, index):
    c = pd.Series(0, index=index)
    for _, row in events.iterrows():
        c.loc[row.t0:row.t1] += 1
    return c

def get_uniqueness(events, concurrency):
    uniq = []
    for _, row in events.iterrows():
        u = (1 / concurrency.loc[row.t0:row.t1]).mean()
        uniq.append(u)
    return pd.Series(uniq, index=events.index)


In [7]:
def sequential_bootstrap(events, uniqueness, n_samples):
    selected = []
    prob = uniqueness / uniqueness.sum()

    for _ in range(n_samples):
        i = np.random.choice(events.index, p=prob)
        selected.append(i)
        prob[i] *= 0.5
        prob /= prob.sum()

    return selected


In [8]:
def get_sample_weights(events, returns, concurrency):
    weights = []
    for _, row in events.iterrows():
        w = (returns.loc[row.t0:row.t1].abs() / 
             concurrency.loc[row.t0:row.t1]).sum()
        weights.append(w)
    return pd.Series(weights, index=events.index)


In [9]:
def apply_time_decay(weights, decay=0.5):
    order = weights.rank()
    decay_weights = np.maximum(0, decay * order + (1 - decay))
    return weights * decay_weights


In [10]:
def shuffle_feature(feature, events):
    shuffled = feature.copy()
    for _, row in events.iterrows():
        window = feature.loc[row.t0:row.t1]
        shuffled.loc[row.t0:row.t1] = np.random.permutation(window.values)
    return shuffled


In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
import numpy as np

def test_feature(feature, events, labels, weights=None):
    # sample feature at event start times
    X = feature.reindex(events["t0"]).values.reshape(-1, 1)
    y = labels.values

    # drop NaN feature rows (early window effects)
    mask = ~np.isnan(X).ravel()
    X = X[mask]
    y = y[mask]
    if weights is not None:
        weights = weights.values[mask]

    clf = DecisionTreeClassifier(max_depth=3)
    clf.fit(X, y, sample_weight=weights)

    preds = clf.predict(X)
    return f1_score(y, preds, average="weighted")


In [12]:
def full_feature_test(feature, close, vol, returns):
    events = get_events(close, vol)
    labels = get_labels(events, close)
    concurrency = get_concurrency(events, close.index)
    uniq = get_uniqueness(events, concurrency)

    base_score = test_feature(feature, events, labels)


    sb_idx = sequential_bootstrap(events, uniq, len(events))
    sb_score = test_feature(
        feature.iloc[sb_idx],
        events.iloc[sb_idx],
        labels.iloc[sb_idx]
    )

    weights = get_sample_weights(events, returns, concurrency)
    weighted_score = test_feature(feature, events, labels, weights)

    

    shuffled = shuffle_feature(feature, events)
    shuffle_score = test_feature(
    shuffled.reindex(events["t0"]),
    events,
    labels
)


    return {
        "base": base_score,
        "seq_bootstrap": sb_score,
        "weighted": weighted_score,
        "shuffled": shuffle_score,
        "avg_uniqueness": uniq.mean()
    }


In [32]:
features = {}

features["ret_1"] = returns
features["ret_5"] = returns.rolling(5).sum()
features["ret_10"] = returns.rolling(10).sum()

features["ret_z"] = (returns - returns.rolling(20).mean()) / returns.rolling(20).std()

features["drawdown"] = close / close.rolling(20).max() - 1

def rolling_slope(series, window):
    out = []
    for i in range(len(series)):
        if i < window:
            out.append(np.nan)
        else:
            y = series.iloc[i-window:i].values
            x = np.arange(window)
            out.append(np.polyfit(x, y, 1)[0])
    return pd.Series(out, index=series.index)

features["trend_slope_10"] = rolling_slope(close, 10)
features["trend_slope_20"] = rolling_slope(close, 20)

features = {}

features["ret_1"] = returns
features["ret_5"] = returns.rolling(5).sum()
features["ret_10"] = returns.rolling(10).sum()

features["ret_z"] = (returns - returns.rolling(20).mean()) / returns.rolling(20).std()

features["drawdown"] = close / close.rolling(20).max() - 1

def rolling_slope(series, window):
    out = []
    for i in range(len(series)):
        if i < window:
            out.append(np.nan)
        else:
            y = series.iloc[i-window:i].values
            x = np.arange(window)
            out.append(np.polyfit(x, y, 1)[0])
    return pd.Series(out, index=series.index)

features["trend_slope_10"] = rolling_slope(close, 10)
features["trend_slope_20"] = rolling_slope(close, 20)

volume = spot["volume"].reindex(common_index)
# Short-term vs long-term realized volatility ratio
rvol_short = returns.rolling(10).std()
rvol_long  = returns.rolling(20).std()

features["vol_expansion"] = rvol_short / rvol_long

features["vol_raw"] = volume
features["vol_z"] = (volume - volume.rolling(20).mean()) / volume.rolling(20).std()
features["vol_trend"] = volume.rolling(5).mean() / volume.rolling(20).mean()
features["vol_ret_interaction"] = volume * returns

funding_aligned = funding["funding_rate"].reindex(common_index).ffill()
oi_aligned = oi["open_interest"].reindex(common_index).ffill()

features["funding_level"] = funding_aligned
features["funding_change"] = funding_aligned.diff()

features["oi_change"] = oi_aligned.pct_change()
features["oi_trend"] = oi_aligned.rolling(5).mean() / oi_aligned.rolling(20).mean()

features["basis_proxy"] = (
    futures["close"].reindex(common_index) - close
)

features["sign_changes"] = returns.rolling(10) \
    .apply(lambda x: np.sum(np.sign(x[1:]) != np.sign(x[:-1])), raw=True)

features["efficiency_ratio"] = (
    close.diff(10).abs() /
    close.diff().abs().rolling(10).sum()
)


In [14]:
for k in features:
    features[k] = features[k].loc[common_index]


In [79]:
# oi[oi['oi_change'] != 'NaN'].tail(33)

,open_interest,oi_change
ts,,
2025-11-27 00:00:00+00:00,NaN,NaN
2025-11-28 00:00:00+00:00,NaN,NaN
2025-11-29 00:00:00+00:00,NaN,NaN
2025-11-30 00:00:00+00:00,88486.551,NaN
2025-12-01 00:00:00+00:00,90584.642,2098.091
2025-12-02 00:00:00+00:00,88384.984,-2199.658
2025-12-03 00:00:00+00:00,88900.526,515.542
2025-12-04 00:00:00+00:00,88324.724,-575.802
2025-12-05 00:00:00+00:00,89253.082,928.358


In [15]:
import pandas as pd
import traceback

rows = []
results = {}

for name, feat in features.items():
    print(f"\n🧪 Testing feature: {name}")

    row = {"feature": name}

    try:
        res = full_feature_test(
            feature=feat,
            close=close,
            vol=vol,
            returns=returns
        )

        row.update(res)
        row["status"] = "ok"
        print(res)

    except Exception as e:
        row["status"] = "fail"
        row["error"] = repr(e)
        row["traceback"] = traceback.format_exc()
        print(f"❌ FAILED feature: {name}")
        print(repr(e))

    rows.append(row)
    
results_df = pd.DataFrame(rows)
results_df.to_csv("feature_test_results.csv", index=False)

results_df



🧪 Testing feature: ret_1
{'base': 0.4079617444679793, 'seq_bootstrap': 0.4003051390012544, 'weighted': 0.4057295728533492, 'shuffled': 0.7103624681038123, 'avg_uniqueness': np.float64(0.23412641559531908)}

🧪 Testing feature: ret_5
{'base': 0.39774076683966775, 'seq_bootstrap': 0.40363281412022567, 'weighted': 0.4062374589956114, 'shuffled': 0.6157776847544268, 'avg_uniqueness': np.float64(0.23412641559531908)}

🧪 Testing feature: ret_10
{'base': 0.4020067907392406, 'seq_bootstrap': 0.4979573804615607, 'weighted': 0.407282446259623, 'shuffled': 0.5925435148702055, 'avg_uniqueness': np.float64(0.23412641559531908)}

🧪 Testing feature: ret_z
{'base': 0.40786114001818835, 'seq_bootstrap': 0.39122392261348843, 'weighted': 0.41555873588620024, 'shuffled': 0.695306599157302, 'avg_uniqueness': np.float64(0.23412641559531908)}

🧪 Testing feature: drawdown
{'base': 0.42615251414442135, 'seq_bootstrap': 0.4267097220752448, 'weighted': 0.4187282164917626, 'shuffled': 0.6159244176461118, 'avg_uni

,feature,base,seq_bootstrap,weighted,shuffled,avg_uniqueness,status,error,traceback
0,ret_1,0.407962,0.400305,0.405730,0.710362,0.234126,ok,NaN,NaN
1,ret_5,0.397741,0.403633,0.406237,0.615778,0.234126,ok,NaN,NaN
2,ret_10,0.402007,0.497957,0.407282,0.592544,0.234126,ok,NaN,NaN
3,ret_z,0.407861,0.391224,0.415559,0.695307,0.234126,ok,NaN,NaN
4,drawdown,0.426153,0.426710,0.418728,0.615924,0.234126,ok,NaN,NaN
5,trend_slope_10,0.460725,0.497878,0.486698,0.491493,0.234126,ok,NaN,NaN
6,trend_slope_20,0.421195,0.449926,0.533102,0.405641,0.234126,ok,NaN,NaN
7,vol_raw,0.403467,0.538743,0.524676,0.405647,0.234126,ok,NaN,NaN
8,vol_z,0.484500,0.449417,0.441631,0.402123,0.234126,ok,NaN,NaN
9,vol_trend,0.405480,0.543370,0.411504,0.422984,0.234126,ok,NaN,NaN


In [27]:
def leakage_test_strict_lag(feature, events, labels, lag=1):
    """
    Shift feature backward so it is guaranteed to be known
    strictly before t0.
    """
    feature_lagged = feature.shift(lag)

    base = test_feature(feature, events, labels)
    lagged = test_feature(feature_lagged, events, labels)

    return {
        "base": base,
        "lagged": lagged,
        "delta": lagged - base
    }

# def leakage_test_horizon_cut(feature, events, labels):
#     clean_feature = feature.copy()

#     for _, row in events.iterrows():
#         clean_feature.loc[row.t0 + pd.Timedelta(days=1) : row.t1] = np.nan

#     return {
#         "base": test_feature(feature, events, labels),
#         "horizon_cut": test_feature(clean_feature, events, labels)
#     }


def leakage_test_jitter(feature, events, labels, jitter=1):
    shifted = feature.copy()

    values = feature.values
    shifts = np.random.choice([-jitter, jitter], size=len(values))

    jittered_values = np.full_like(values, np.nan)

    for i, s in enumerate(shifts):
        j = i + s
        if 0 <= j < len(values):
            jittered_values[i] = values[j]

    shifted[:] = jittered_values

    return {
        "base": test_feature(feature, events, labels),
        "jittered": test_feature(shifted, events, labels)
    }



In [56]:
events = get_events(
    close=close,
    vol=vol,
    pt_sl=(1, 1),
    max_holding=20
)
labels = get_labels(events, close)
print(f"✅ Generated {len(events)} events")
# print(events.head())

suspects = {
    "funding_level": features["funding_level"],
    "funding_change": features["funding_change"],
    "oi_change": features["oi_change"]
}

for name, feat in suspects.items():
    print(f"\n🔍 Leakage audit: {name}")

    print("Strict lag:", leakage_test_strict_lag(feat, events, labels))
    # print("Horizon cut:", leakage_test_horizon_cut(feat, events, labels))
    print("Jitter:", leakage_test_jitter(feat, events, labels))


✅ Generated 3034 events

🔍 Leakage audit: funding_level
Strict lag: {'base': 0.6137240263780018, 'lagged': 0.7857789855072465, 'delta': 0.17205495912924462}
Jitter: {'base': 0.6137240263780018, 'jittered': 0.6527510265024262}

🔍 Leakage audit: funding_change
Strict lag: {'base': 0.7096642738472149, 'lagged': 0.765993265993266, 'delta': 0.05632899214605114}
Jitter: {'base': 0.7096642738472149, 'jittered': 0.8058229352346998}

🔍 Leakage audit: oi_change
Strict lag: {'base': 0.882051282051282, 'lagged': 0.8589743589743589, 'delta': -0.023076923076923106}
Jitter: {'base': 0.882051282051282, 'jittered': 0.8589743589743589}


In [63]:
def filter_events_by_feature(events, feature):
    valid_t0 = feature.dropna().index
    mask = events["t0"].isin(valid_t0)
    return events.loc[mask]

def regime_bucket_analysis(feature, events, close, n_bins=5):
    """
    Split events into feature quantiles and evaluate outcome distributions.
    """

    # 1. Filter events where feature is available at t0
    events_f = filter_events_by_feature(events, feature)

    if len(events_f) == 0:
        return {"status": "degenerate", "samples": 0}

    # 2. Sample feature at event start times
    feature_at_t0 = feature.reindex(events_f["t0"])

    # 3. Build analysis DataFrame
    df = pd.DataFrame({
        "feature": feature_at_t0.values,
        "t0": events_f["t0"].values,
        "t1": events_f["t1"].values
    }).dropna()

    if len(df) == 0:
        return {"status": "degenerate", "samples": 0}

    # 4. Compute event returns
    df["ret"] = (
        close.reindex(df["t1"]).values /
        close.reindex(df["t0"]).values - 1
    )

    # 5. Bucket by feature quantiles
    df["bucket"] = pd.qcut(df["feature"], q=n_bins, labels=False)

    # 6. Aggregate statistics
    summary = df.groupby("bucket")["ret"].agg(
        count="count",
        avg_ret="mean",
        vol="std",
        hit_rate=lambda x: (x > 0).mean()
    )

    return summary




In [64]:
regime_candidates = {
    "funding_level": features["funding_level"],
    "vol_expansion": features["vol_expansion"],
    "basis_proxy": features["basis_proxy"]
}

for name, feat in regime_candidates.items():
    print(f"\n📊 Regime analysis: {name}")
    print(regime_bucket_analysis(feat, events, close))



📊 Regime analysis: funding_level
        count  avg_ret  vol  hit_rate
bucket                               
0           0      NaN  NaN       0.0
1           0      NaN  NaN       0.0
2           0      NaN  NaN       0.0
3           0      NaN  NaN       0.0
4           0      NaN  NaN       0.0

📊 Regime analysis: vol_expansion
        count  avg_ret  vol  hit_rate
bucket                               
0           0      NaN  NaN       0.0
1           0      NaN  NaN       0.0
2           0      NaN  NaN       0.0
3           0      NaN  NaN       0.0
4           0      NaN  NaN       0.0

📊 Regime analysis: basis_proxy
        count  avg_ret  vol  hit_rate
bucket                               
0           0      NaN  NaN       0.0
1           0      NaN  NaN       0.0
2           0      NaN  NaN       0.0
3           0      NaN  NaN       0.0
4           0      NaN  NaN       0.0


In [54]:
def return_attribution_test(feature, events, close, labels, n_bins=10):
    feature_at_t0 = feature.reindex(events["t0"])
    events_f = filter_events_by_feature(events, feature)
    if len(events_f) < 50:
        return {"status": "degenerate", "samples": len(events_f)}

       # build df first
    df = pd.DataFrame({
        "feature": feature.reindex(events["t0"]).values,
        "t0": events["t0"].values,
        "t1": events["t1"].values
    }).dropna()
    
    # now compute prices aligned to df
    p0 = close.reindex(df["t0"]).values
    p1 = close.reindex(df["t1"]).values
    
    valid = ~np.isnan(p0) & ~np.isnan(p1)
    
    df = df.loc[valid]
    
    df["abs_ret"] = np.abs(p1[valid] / p0[valid] - 1)
    if df["abs_ret"].nunique() < n_bins:
        return {
            "status": "degenerate",
            "unique_abs_ret": df["abs_ret"].nunique(),
            "samples": len(df)
        }

    df["decile"] = pd.qcut(
        df["abs_ret"],
        q=n_bins,
        labels=False,
        duplicates="drop"
    )

    scores = []

    for d in range(n_bins):
        idx = df[df["decile"] == d].index
        if len(idx) < 20:
            scores.append(np.nan)
            continue

        sub_events = events.loc[idx]
        sub_labels = labels.loc[idx]

        score = test_feature(feature, sub_events, sub_labels)
        scores.append(score)

    return pd.Series(scores, index=range(n_bins))


In [55]:
return_candidates = {
    "trend_slope_10": features["trend_slope_10"],
    "trend_slope_20": features["trend_slope_20"],
    "drawdown": features["drawdown"]
}

for name, feat in return_candidates.items():
    print(f"\n📈 Return attribution: {name}")
    print(return_attribution_test(feat, events, close, labels))



📈 Return attribution: trend_slope_10
{'status': 'degenerate', 'samples': 0}

📈 Return attribution: trend_slope_20
{'status': 'degenerate', 'samples': 0}

📈 Return attribution: drawdown
{'status': 'degenerate', 'samples': 0}
